In [90]:
import datetime
import logging
import os
import re
import shutil
import sys
from datetime import date

import arcpy
from arcgis.gis import GIS

In [91]:
gis = GIS("home")
user = gis.users.me

arcgis_project = arcpy.mp.ArcGISProject("CURRENT")
active_map = arcgis_project.activeMap

UTILITY = "ATCO"
LAC_AREAS_ALL = ["Calgary"]
LAYER_TYPE = "NETWORK"
MAP_TYPE = "Vector Tile Service"

In [92]:
PACKAGE_DIR = r"C:\Users\sameer.bajracharya\Sam_works\DEV\ArcGIS_Enterprise\VTPK" # Update the directory to folder location
AGOL_FOlDER_NAME = "VTPK" # Update the name of the folder from ArcGIS Enterprise/online
SET_WKID = 3857 # The WKID for WGS_1984_Web_Mercator_Auxiliary_Sphere is 3857

# Set Layer Scales based on Layer name (e.g Streets)
LAYER_TEXT = "STREETS"

LABEL_MIN_SCALE = 10000
LABEL_MAX_SCALE = 1

# Set scales here for Vector Tile Package layer
MIN_CACHED_SCALE = 577791 # 1155581, 577791, 288895 or 144448
MAX_CACHED_SCALE = 282 # 564, 282 or 141

LAYERS_TO_SKIP = [
    "WORLD TOPOGRAPHIC MAP",
    "WORLD HILLSHADE",
    "BASEMAP"
]

In [93]:
# Configure the log file
# Get the directory where the script is located
try:
    current_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    # If __file__ fails, we are likely in a Notebook or Console
    current_dir = os.getcwd()

# Create the full path for the log file
log_filename = f"VTPK_Creation_Updates_{date.today()}.log"
log_path = os.path.join(current_dir, log_filename)

# Force a fresh handle on the log file (especially useful in Notebooks)
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

logging.basicConfig(
    filename=log_path,
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    filemode='a'
)

In [94]:
# Finds all unique codes in the layer names within a given map   
def Extract_Map_Code(name_list):
    """
    Parses a list of layer names to extract unique codes (suffixes).
    Assumes codes are the segment after the last underscore (e.g., 'Data_Layer_ON01' -> 'ON01').

    Args:
        name_list (list): A list of strings (layer names).

    Returns:
        set: A set of unique codes found.
    """
    if not name_list:
        return set()
        
    # set() automatically removes duplicates
    unique_codes = {name.split('_')[-1] for name in name_list if '_' in name}
    
    # Handle case where no underscores existed
    if not unique_codes and name_list:
        print("Warning: No underscores found in layer names.")

    return unique_codes

print("Completed running function 'Extract Layer Codes'")

Completed running function 'Extract Layer Codes'


In [95]:
def Search_AGOL(query, feature_type):
    try:
        # Ensure we use the exact title field in the query string
        search_text = f'title:"{query}" AND type:"{feature_type}"'
        print(f"Searching for: {search_text}")
        
        # max_items=1 is risky if there are duplicates, so 10 is a safe 'short list'
        search_results = gis.content.search(
            query=search_text, 
            max_items=10
        )

        # Use a generator expression with 'next' for a faster, cleaner exit
        # This replaces the manual for-loop
        match = next((item for item in search_results if item.title == query), None)

        if match:
            return match.title, match.id
        
        print(f"    {query} - No exact title match found.")
        return None
    
    except Exception as e:
        print(f"Error Searching Layers: {e}")
        return None

In [96]:
def Set_Layer_Scale(layer_lists):
    if not layer_lists:
        return

    print(f"Found {len(layer_lists)} layers...")

    for layer in layer_lists:
        try: 
            # Normalize name to UPPERCASE for robust matching
            lyr_name = layer.name.upper()
            
            # Check Skip List first (Fast O(1) Lookup)
            if lyr_name in LAYERS_TO_SKIP:
                # Optional feature: Auto-turn off basemaps if they are on
                if layer.visible: 
                    print(f"Skipping & Hiding Base Map: {layer.name}")
                    layer.visible = False 
                else:
                    print(f"Skipping: {layer.name}")
                continue # This moves to the next layer in the loop

            if layer.supports("minThreshold") and layer.supports("maxThreshold"):
                # print(f"Setting scale for: {layer.name}")
                # layer.minThreshold = default_min_scale
                # layer.maxThreshold = default_max_scale

                if LAYER_TEXT.lower() in layer.name.lower():
                    print(f"Setting scale for: {layer.name} with {LABEL_MIN_SCALE}")
                    layer.minThreshold = LABEL_MIN_SCALE
                    layer.maxThreshold = LABEL_MAX_SCALE
                # else:
                #     print(f"Setting scale for: {layer.name} with {default_min_scale}")
                #     layer.minThreshold = default_min_scale
                #     layer.maxThreshold = default_max_scale
            else:
                print(f"\nSkipping Layer: {layer.name}\n")

            # If the layer is a group layer, recursively call this function
            # if layer.isGroupLayer:
            #     print(f"\n--- Entering Group: {layer.name} ---\n")
            #     set_layer_scale(layer.listLayers())
                # print(f"--- Exiting Group: {layer.name} ---")

        except Exception as e:
            print(f"Error setting scale for {layer.name}: {e}")
    

In [97]:
def Create_Thumbnail(arcgis_project, active_map, valid_layers):
    # Set Extent & Zoom
    found_valid_layer = False 
    map_view = arcgis_project.activeView
    
    # print("\nSearching for data extent...")
    # Reset environments first to ensure no old settings persist
    arcpy.ClearEnvironment("extent")
    
    for layer in valid_layers:
        # Skip Group Layers and check Datasource support ---
        # Group layers do not have a standard 'datasource' and can cause extent errors
        if layer.isGroupLayer:
            continue

        if layer.supports("DATASOURCE") and layer.visible:
            try:
                # Use arcpy.Describe to get the extent (Works on all versions)
                desc = arcpy.Describe(layer)
                
                if not desc.extent or desc.extent.width == 0:
                    print(f"Skipping {layer.name}: Extent is empty.")
                    continue

                # Project the extent to match the Map's Spatial Reference
                projected_extent = desc.extent.projectAs(active_map.spatialReference)
                
                # Check if extent is valid (has width/height)
                if projected_extent.width > 0 :
                    # This stops the tool from guessing (and guessing wrong).
                    arcpy.env.extent = projected_extent
                    arcpy.env.outputCoordinateSystem = active_map.spatialReference
                    
                    print(f"Environment extent set to >>> {layer.name}")
                    
                    if map_view:
                        map_view.camera.setExtent(projected_extent)
                    
                    # print(f"Extent set from layer: {layer.name}")
                    found_valid_layer = True
                    break
                
            except Exception as e:
                # If a specific layer fails (e.g., query layer issues), print and continue to next
                print(f"Skipping layer {layer.name} due to extent error: {e}")
                continue

    if not found_valid_layer:
        print("Warning: No valid data layers found to set extent.")

    # Export Thumbnail
    if map_view:
        thumb_name = f"VTPK_{UTILITY}_{LAYER_TYPE}_{active_map.name}.png"
        thumb_path = os.path.join(PACKAGE_DIR, thumb_name)
        
        print(f"Exporting thumbnail to: {thumb_path}")
        
        try:
            if 'projected_extent' in locals():
                map_view.camera.setExtent(projected_extent)
            
            # 2 seconds is usually enough; increase to 5 if layers are heavy
            print("Waiting for map rendering...")
            time.sleep(3) 

            # Export
            map_view.exportToPNG(thumb_path, width=600, height=400)
            
            # Check if file size is too small (indicates blank image)
            if os.path.exists(thumb_path) and os.path.getsize(thumb_path) < 5000:
                print("! Warning: Thumbnail seems suspiciously small (likely blank)")
        except Exception as e:
            print(f"Warning: Could not export thumbnail (View might not be ready). {e}")
    
    

In [98]:
def Create_VTPK(indx_feature, output_vtpk, output_feature, active_map):
    try:
        
        # --- Record the start time ---
        script_start_time = time.perf_counter()
        print(f"\nStarting to create vector tile index for the map: '{active_map.name}'...")

        # print("Checking map metadata...")
        map_metadata = active_map.metadata
        
        # Set the Summary/Description and Tags (both are usually required for VTPKs)
        map_metadata.summary = "Vector Tile Package generated via Python script."
        map_metadata.tags = "VTPK, Automated"
        map_metadata.description = active_map.name
        
        # Save the metadata back to the map object
        active_map.metadata = map_metadata
    
        # Execute the Create Vector Tile Index tool using the active map as input.
        arcpy.management.CreateVectorTileIndex(
            in_map=active_map,
            # out_featureclass=r"C:\Users\sameer.bajracharya\Sam_works\DEV\ArcGIS_Enterprise\OW01_VTPK_1M_500_test.shp",
            out_featureclass=output_feature,
            service_type="ONLINE",
            tiling_scheme=None,
            vertex_count=10000,
        )
    
        print(f"Successfully created index features at: {output_feature}\n")
    
        # Remove the tile index layer from the layerlist before creating the vector tile package.
        remove_layer_name = indx_feature
    
        layer_to_remove = None
        
        for lyr in active_map.listLayers():
            if lyr.name == remove_layer_name:
                layer_to_remove = lyr
                break
        
        if layer_to_remove:
            active_map.removeLayer(layer_to_remove)
            print(f"Successfully removed layer from content: '{remove_layer_name}'\n")
        else:
            print(f"Error: Layer '{remove_layer_name}' not found in the map '{active_map.name}'.")
    
        # Creating a vector tile package using the index feature 
        print(f"Starting to Create Vector Tile Package : '{active_map.name}'...")

        arcpy.management.CreateVectorTilePackage(
            in_map=active_map,
            output_file= output_vtpk,
            service_type="ONLINE",
            tiling_scheme=None,
            tile_structure="INDEXED",
            min_cached_scale=577791, #1155581, 577791, 288895 or 144448
            max_cached_scale=282, #564 or 282 or 141
            # index_polygons=r"C:\Users\sameer.bajracharya\Sam_works\DEV\ArcGIS_Enterprise\Alektra_HN01_Network_VTPK_Index_Feature.shp",
            index_polygons= output_feature,
            summary="Vector Tile Package Created using python script.",
            tags="VTPK"
        )

        print(f"Successfully created Vector Tile Package at: {output_vtpk}")
        return output_vtpk

    except arcpy.ExecuteError:
        # Get and print geoprocessing error messages.
        print(arcpy.GetMessages(2))
    
    except Exception as e:
        print(f"\nAn unexpected error occurred: {e}")
        error_msg = f"ERROR: Failed to update LAC: {active_map.name}. Reason: {str(e)}"
        print(error_msg)
        logging.error(error_msg)
    
    finally:
        # --- 3. This block always runs (success or error) ---
        script_end_time = time.perf_counter()
        elapsed_time = script_end_time - script_start_time
        
        # Optional: Convert to minutes and seconds for readability
        minutes = int(elapsed_time // 60)
        seconds = elapsed_time % 60
        
        print(f"{'='*50}")
        print(f"Script finished in {minutes} minutes and {seconds:.2f} seconds.")
        print(f"(Total elapsed: {elapsed_time:.2f} seconds)")
        success_msg = f"\n\nSUCCESS: Script finished in {minutes} minutes and {seconds:.2f} seconds. File path: {output_vtpk}"
        logging.info(success_msg)
    

In [99]:
def Process_Map(arcgis_project, active_map, layer_status):
    print(f"{'='*50}")
    try:
        if not active_map:
            print("Active view is not a Map. Searching project for maps...")
            maps = arcgis_project.listMaps()
            if len(maps) > 0:
                active_map = maps[0] # Default to the first map in the project
                print(f"Defaulting to first map: {active_map.name}")
            else:
                raise Exception("No maps found in the project.")
        else:
            print(f"Active Map is: {active_map.name}")
            print(f"Log file is located at: {log_path}")

        try:
            update_coordinate_system = arcpy.SpatialReference(SET_WKID)
            active_map.spatialReference = update_coordinate_system
            print(f"Spatial Reference set to: {active_map.spatialReference.name}")
            
        except Exception as e:
            print(f"Warning: Could not set Spatial Reference. {e}")

        # Check for Broken Layers ---
        all_layers = active_map.listLayers()

        basemap_names = ["World Topographic Map", "World Hillshade"]
        
        # Filter for operational layers (not broken AND not a standard basemap)
        operational_data = [
            lyr for lyr in all_layers 
            if not lyr.isBroken and lyr.name not in basemap_names
        ]

        if not operational_data:
            print("!!!" + "="*50)
            print("VALIDATION FAILED: No operational layers found.")
            print("Map only contains Basemaps or broken layers.")
            print("Exiting script to prevent empty VTPK generation.")
            print("!!!" + "="*50)
            sys.exit(0)
        
        # Create a list of ONLY valid layers (excludes broken links)
        valid_layers = [l for l in all_layers if not l.isBroken]
    
        if not valid_layers:
            raise Exception("No valid (unbroken) layers found in the active map.")
        
        if len(valid_layers) < len(all_layers):
            print(f"Warning: {len(all_layers) - len(valid_layers)} broken layers were skipped.")
    
        # Apply Scaling Rules
        try:
            Set_Layer_Scale(valid_layers)
            # print("----- Scale Updated -----")        
        except Exception as e:
            print(f"! Warning: Issue in scaling function: {e}")

        # Rename Map based on First Valid Layer
        first_layer_name = valid_layers[0].name
        print(f"{'='*50}")
        print(f"\nFirst Layer: {first_layer_name}")
    
        map_codes = Extract_Map_Code([first_layer_name]) 
        
        if map_codes:
            new_map_name = list(map_codes)[0]
            print(f"Renaming map from '{active_map.name}' to '{new_map_name}'...")
            active_map.name = new_map_name
        else:
            print("Warning: Could not extract code from first layer. Map name unchanged.")

        Create_Thumbnail(arcgis_project, active_map, valid_layers)

         # Set Workspace
        workspace = arcgis_project.defaultGeodatabase
        if workspace and arcpy.Exists(workspace):
            arcpy.env.workspace = workspace
            print(f"Workspace set to: {arcpy.env.workspace}")
        else:
            print("Warning: Default geodatabase not found.")
    
        # Determine the prefix based on status
        prefix = "UPDATED_VTPK" if layer_status != "True" else "VTPK"
        
        # Construct base name once
        base_identity = f"{prefix}_{UTILITY}_{LAYER_TYPE}_{active_map.name}"
        
        # Generate final variables
        output_index_feature = f"{base_identity}_Tile_Index"
        vtpk_name = f"{base_identity}.vtpk"
        
        output_vtpk_package = os.path.join(PACKAGE_DIR, vtpk_name)
        out_feature_path = os.path.join(workspace, output_index_feature)            
        
        print(f"{'='*50}")
        print(f"Ready to Process: {output_index_feature}")
        print(f"Output VTPK:      {output_vtpk_package}")
        print(f"{'='*50}")

        return Create_VTPK(output_index_feature, output_vtpk_package, out_feature_path, active_map)

    except SystemExit:
        # This catches the sys.exit() so the shell doesn't report it as a crash
        pass
    except Exception as e:
        print(f"Error accessing the project or active map: {e}")
        # Return None or sys.exit(0) instead of using the bare exit()
        return None
        

In [100]:
def Publish_VTPK(vtpk_path):
    try:        
        # Set the item properties for the new item in your portal
        item_properties = {
            "title": f"{UTILITY}_{LAYER_TYPE}_{active_map.name}_VTPK",
            "tags": f"Vector, Tiles, {active_map.name}, {LAYER_TYPE}, {UTILITY}",
            "type": "Vector Tile Package",
            "description": f"{UTILITY} {LAYER_TYPE} {active_map.name} Hosted Vector Tile Layers."
        }
    
        # Add the .vtpk to your portal as an item
        # print("Uploading the .vtpk package...")
        set_folder = gis.content.folders.get(AGOL_FOlDER_NAME)

        vtpk_item = set_folder.add(item_properties,
                                   file=vtpk_path)
        
        while not vtpk_item.done():
            print("...Uploading/Processing VTPK...")
            time.sleep(3)
            
        new_item = vtpk_item.result()
        print(f"VTPK uploaded Successfully: {new_item.title}")
        
        # # Publish the item to create the hosted layer
        # print("Publishing hosted vector tile layer...")
        hosted_tile_layer_item = new_item.publish()
        logging.info(f"Published Tile Layer URL: {hosted_tile_layer_item.url}")
        
        return (f"\nSuccessfully Published Tile layer: {hosted_tile_layer_item.url}")
        
    except Exception as e:
        print(f"Error in Publish_VTPK: {e}")
        return None
        

In [101]:
def Cleanup_Archives(title, keep_count=1):
    """
    Searches for and deletes older archive items, keeping a specified number as backups.
    """
    # 1. ADD OWNER CONSTRAINT: Prevent accidental deletion of other users' items
    owner = gis.users.me.username
    # ArcGIS Lucene queries can be finicky with quotes and wildcards. 
    # This structure is generally safer for partial matches.
    archive_query = f"title:{title}_archive* AND owner:{owner}"
    print(f"{'='*50}")
    print(f"Searching for archives using query: {archive_query}...")
    
    # 2. INCREASE MAX ITEMS: Ensure we don't miss ancient archives if they pile up
    # Note: item_type is correctly set here!
    archives = gis.content.search(
        query=archive_query, 
        item_type="Vector Tile Layer", 
        max_items=10
    )
    
    if not archives:
        print("No archives found to clean up.")
        return

    # Sort the list of archives by their creation date (newest first)
    # Note: item.created is a Unix timestamp in milliseconds
    archives.sort(key=lambda item: item.created, reverse=True)
    
    # Check if we have more archives than we want to keep
    if len(archives) <= keep_count:
        print(f"Found {len(archives)} archive(s). Keeping all of them.")
    else:
        items_to_delete = archives[keep_count:]
        print(f"Found {len(archives)} archive(s). Deleting the oldest {len(items_to_delete)}...")
        
        for item in items_to_delete:
            # 3. THE FINAL SAFETY NET: Trust, but verify.
            if item.type == "Vector Tile Layer":
                print(f" -> Deleting: {item.title} (ID: {item.id})")
                item.delete()
            else:
                print(f" -> WARNING: Skipped deleting {item.title}. Expected a Vector Tile Layer but found {item.type}!")
            
        print("Archive cleanup complete.")

In [102]:
def Find_Source_VTPK(vtl_item):
    """
    Finds the original source file (VTPK) used to publish a hosted layer.
    """
    print(f"Searching for the source VTPK for layer: {vtl_item.title}...")
    
    # 'Service2Data' looks for the data item that powers the service
    related_items = vtl_item.related_items("Service2Data", direction="forward")
    
    if related_items:
        for source_item in related_items:
            # CRITICAL SAFETY CHECK: Explicitly verify the item type
            if source_item.type == "Vector Tile Package":
                print(f"Found source VTPK: {source_item.title} (ID: {source_item.id})")
                return source_item
                
        # If the loop finishes without returning, the related item was the wrong type
        print(f"Warning: Found related items for {vtl_item.title}, but none were Vector Tile Packages.")
        return None
        
    else:
        print(f"Warning: Could not find any related source data for {vtl_item.title}.")
        return None

In [103]:
def Update_Publish_VTPK(vtpk_path, title, item_id):
    try:
        # Get the live hosted layer you want to update
        target_item = gis.content.get(item_id)
        
        # FIND THE OLD VTPK BEFORE THE SWAP
        old_vtpk = Find_Source_VTPK(target_item)
        
        # Define the clean, final name for the UI
        final_vtpk_title = f"{UTILITY}_{LAYER_TYPE}_{active_map.name}_VTPK"
        
        item_properties = {
            "title": f"STAGING_{final_vtpk_title}", # Temporary UI name
            "tags": f"Vector, Tiles, {active_map.name}, {LAYER_TYPE}, {UTILITY}",
            "type": "Vector Tile Package",
            "description": f"{UTILITY} {LAYER_TYPE} {active_map.name} Hosted Vector Tile Layers."
        }

        set_folder = gis.content.folders.get(AGOL_FOlDER_NAME)
        
        # ---------------------------------------------------------
        # CREATE A UNIQUE LOCAL FILENAME 
        # ---------------------------------------------------------
        print("Creating a uniquely named local copy for upload...")
        timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        base_dir = os.path.dirname(vtpk_path)
        base_name = os.path.basename(vtpk_path)
        name, ext = os.path.splitext(base_name)
        
        unique_vtpk_path = os.path.join(base_dir, f"{name}_{timestamp}{ext}")
        
        # Copy the original file to the new unique path
        shutil.copy2(vtpk_path, unique_vtpk_path)

        # ---------------------------------------------------------
        # 2. UPLOAD & PUBLISH STAGING
        # ---------------------------------------------------------
        print("Uploading new staging VTPK...")
        staging_vtpk_future = set_folder.add(item_properties, file=unique_vtpk_path)
        staging_vtpk = staging_vtpk_future.result()
        
        # Clean up the local temp file to save disk space
        if os.path.exists(unique_vtpk_path):
            os.remove(unique_vtpk_path)

        unique_service_name = f"{final_vtpk_title}_{timestamp}"

        publish_params = {
            "name": unique_service_name,  # Backend service name (must be unique)
            "title": final_vtpk_title     # Frontend UI name
        }

        print("Publishing staging VTL...")
        staging_vtl = staging_vtpk.publish(
            publish_parameters=publish_params, 
            overwrite=True
        )

        # ---------------------------------------------------------
        # REPLACE THE LIVE LAYER
        # ---------------------------------------------------------
        print("Replacing target layer with new staging layer...")
        gis.content.replace_service(
            replace_item=target_item, 
            new_item=staging_vtl,
            replace_metadata=False 
        )

        # ---------------------------------------------------------
        # SAFELY DELETE THE ARCHIVED SERVICE
        # ---------------------------------------------------------
        print("Deleting the archived layer (old service) to release locks...")
        try:
            # Re-fetch to ensure we aren't using a stale object
            refresh_staging_vtl = gis.content.get(staging_vtl.id)
            if refresh_staging_vtl:
                refresh_staging_vtl.delete()
        except Exception as e:
            print(f" -> Non-fatal warning: Could not delete staging VTL. ({e})")

        # ---------------------------------------------------------
        # PRE-FLIGHT CHECK: Remove duplicate VTPKs safely
        # ---------------------------------------------------------
        print(f"Checking for existing VTPKs named {final_vtpk_title}...")
        existing_items = gis.content.search(
            query=f"title:'{final_vtpk_title}' AND owner:{gis.users.me.username}",
            item_type="Vector Tile Package"
        )
        
        old_vtpk_id = old_vtpk.id if old_vtpk else None
        
        for item in existing_items:
            # CRITICAL: Do not delete the one we just uploaded OR the old source item yet!
            if item.id != staging_vtpk.id and item.id != old_vtpk_id:
                if item.type == "Vector Tile Package":
                    try:
                        print(f"Removing duplicate VTPK: {item.title} ({item.id})")
                        verify_item = gis.content.get(item.id)
                        if verify_item:
                            verify_item.delete()
                    except Exception as e:
                        print(f" -> Warning: Failed to delete duplicate {item.title}: {e}")

        # ---------------------------------------------------------
        # MAKE STAGING VTPK THE NEW OFFICIAL VTPK
        # ---------------------------------------------------------
        print("Renaming staging VTPK to become the official source...")
        staging_vtpk.update(item_properties={"title": final_vtpk_title})

        # ---------------------------------------------------------
        # SAFELY CLEAN UP THE OLD VTPK
        # ---------------------------------------------------------
        if old_vtpk and old_vtpk.id != staging_vtpk.id:
            print(f"Deleting old source VTPK: {old_vtpk.title}...")
            try:
                # Re-fetch because the object is old and stale by this point
                verify_old_vtpk = gis.content.get(old_vtpk.id)
                if verify_old_vtpk:
                    verify_old_vtpk.delete()
                else:
                    print(" -> Old VTPK was already removed or is inaccessible.")
            except Exception as e:
                print(f" -> Non-fatal warning: Could not delete old VTPK. ({e})")
        
        # ---------------------------------------------------------
        # CLEANUP ARCHIVES
        # ---------------------------------------------------------
        Cleanup_Archives(title=title, keep_count=1)
        
        print("Success: Layer updated and names synchronized.")
        
    except Exception as e:
        print(f"An error occurred in Update_Publish_VTPK: {e}")

In [104]:
# def Update_Publish_VTPK(vtpk_path, title, item_id):
#     try:
#         target_item = gis.content.get(item_id)
        
#         item_properties = {
#             "title": f"TEST_{UTILITY}_{LAYER_TYPE}_{active_map.name}_VTPK_STAGING",
#             "tags": f"Vector, Tiles, {active_map.name}, {LAYER_TYPE}, {UTILITY}",
#             "type": "Vector Tile Package",
#             "description": f"{UTILITY} {LAYER_TYPE} {active_map.name} Hosted Vector Tile Layers."
#         }
        
#         set_folder = gis.content.folders.get(AGOL_FOlDER_NAME)
        
#         staging_vtpk_future = set_folder.add(item_properties,
#                                              file=vtpk_path)
        
#         staging_vtpk = staging_vtpk_future.result()

#         staging_vtl = staging_vtpk.publish(overwrite=True)

#         # target_item.manager.replace(
#         #     replace_with_item=staging_vtl,
#         #     archive_item_name=f"{title}_ARCHIVE")

#         gis.content.replace_service(
#             replace_item=target_item, 
#             new_item=staging_vtl,
#             replace_metadata=False # Change to True if you want the new VTPK's tags/summary to overwrite the old ones
#         )
        
#         # Cleanup staging items
#         staging_vtl.delete()
#         staging_vtpk.delete()
#         print("Layer successfully updated via replacement.")
        
#     except Exception as e:
#         print(f"An error occurred: {e}")

        

In [105]:
try:
    print(f"Successfully connected to {gis.properties.portalName} as {gis.properties.user.username}")
    # Search for Feature Layers (Hosted Feature Services)
    # print("\n--- Searching for Layers ---")

    for lac in LAC_AREAS_ALL:
        # print(f"{UTILITY}_{LAYER_TYPE}_{lac}_VTPK")                   
        search_VTPK_layer = Search_AGOL(f"{UTILITY}_{LAYER_TYPE}_{lac}_VTPK", MAP_TYPE) # ALECTRA_NETWORK_GW01_VTPK        
        
        
        if search_VTPK_layer:
            title, item_id = search_VTPK_layer
            print(f"Found in AGOL, So Updating existing data...")
            print(f"Found Data: {title} (ID: {item_id})")
            
            update_data = Process_Map(arcgis_project, active_map, layer_status="False")
            print(f"{'='*50}")
            print(f"Data:{update_data}")
            logging.info(f"UPDATING : {title} (ID: {item_id})")
            Update_Publish_VTPK(update_data, title, item_id)
            
        else:
            print(f"Not Found in AGOL, So Publishing New")
            logging.info(f"CREATING : {UTILITY}_{LAYER_TYPE}_{lac}_VTPK")
            create_data = Process_Map(arcgis_project, active_map, layer_status="True")
            print(f"{'='*50}")
            print(f"Data:{create_data}")
            
            Publish_VTPK(create_data)
        # Function to update the webmap
        # Update_Webmap(gis, lac, webmap_update, general_landbase)          
        
except Exception as e:
    print(f"An error occurred: {e}")

Successfully connected to ArcGIS Enterprise as sameer.bajracharya@planview.ca
Searching for: title:"ATCO_NETWORK_Calgary_VTPK" AND type:"Vector Tile Service"
    ATCO_NETWORK_Calgary_VTPK - No exact title match found.
Not Found in AGOL, So Publishing New
Active Map is: Calgary
Log file is located at: C:\Users\sameer.bajracharya\Documents\ArcGIS\Projects\Create_Vector_Tile_Package\VTPK_Creation_Updates_2026-03-25.log
Spatial Reference set to: WGS_1984_Web_Mercator_Auxiliary_Sphere
Found 46 layers...
Skipping: World Topographic Map
Skipping: World Hillshade

First Layer: ATCO_NETWORK_Calgary
Renaming map from 'Calgary' to 'Calgary'...
Environment extent set to >>> CP_TEST_POINT
Exporting thumbnail to: C:\Users\sameer.bajracharya\Sam_works\DEV\ArcGIS_Enterprise\VTPK\VTPK_ATCO_NETWORK_Calgary.png
Waiting for map rendering...
! Warning: Thumbnail seems suspiciously small (likely blank)
Workspace set to: C:\Users\sameer.bajracharya\Documents\ArcGIS\Projects\Create_Vector_Tile_Package\Create_